# Data Engineer Flood Claims Pipeline
* End-to-end pipeline analyzing real flood claims FEMA data. 
* Current stack: Python, AWS S3, AWS Glue, PySpark, Docker, and OpenTofu.
* Planned extensions: dbt, DuckDB, BigQuery, Kafka, and Kestra.

In [1]:
import boto3
from kaggle_secrets import UserSecretsClient

print("Step 1: Loading Secrets")
secrets = UserSecretsClient()

AWS_ACCESS_KEY_ID = secrets.get_secret("FEMA_ACCESS_KEY").strip()
AWS_SECRET_ACCESS_KEY = secrets.get_secret("FEMA_SECRET_KEY").strip()
BUCKET_NAME = secrets.get_secret("FEMA_S3_BUCKET").strip()
AWS_REGION = secrets.get_secret("AWS_REGION").strip()

print("Step 2: Creating AWS session")

session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)

print("Step 3: Testing AWS identity")

try:
    sts = session.client("sts")
    identity = sts.get_caller_identity()
    print("PASSED: AWS identity connected")
except Exception as e:
    print("FAILED: AWS identity test")
    print(e)

print("Step 4: Testing S3 bucket access")

try:
    s3 = session.client("s3")
    response = s3.list_objects_v2(Bucket=BUCKET_NAME, MaxKeys=5)
    print("PASSED: S3 bucket access")
    print("Objects found:", response.get("KeyCount", 0))
except Exception as e:
    print("FAILED: S3 bucket access")
    print(e)

Step 1: Loading Secrets
Step 2: Creating AWS session
Step 3: Testing AWS identity
PASSED: AWS identity connected
Step 4: Testing S3 bucket access
PASSED: S3 bucket access
Objects found: 5


In [2]:
import requests
import json
import pandas as pd
import hashlib

base_url = "https://www.fema.gov/api/open/v2/FimaNfipClaims"
s3_key = "raw/fema/unprocessed/fema_claims.json"

headers = {
    "User-Agent": "Scott Schmidt FEMA NFIP claims data engineering project",
    "Accept": "application/json"
}

# Creates a dictionary of API query parameters:
params = {
    "$top": 100,
    "$skip": 0
}

response = requests.get(
    base_url,
    headers=headers,
    params=params,
    )

print("claims ", type(response))

claims_string = json.dumps(response.json(), sort_keys=True)
claims_hash = hashlib.md5(claims_string.encode()).hexdigest()

curr_s3_claims_file = s3.get_object(Bucket=BUCKET_NAME, Key=s3_key)
curr_s3_bytes = curr_s3_claims_file["Body"].read()
curr_hash = hashlib.md5(curr_s3_bytes).hexdigest()

if claims_hash == curr_hash:
    print("Stop Program. File has not changed.")
    exit()

if response.status_code == 200:
    print("URL used:", response.url)
    print("Status code:", response.status_code)
    print("Content type:", response.headers.get("Content-Type"))
    print(response.text[:300])
else:
    print("Request failed:", response.status_code)
    print(response.text)

claims  <class 'requests.models.Response'>
URL used: https://www.fema.gov/api/open/v2/FimaNfipClaims?%24top=100&%24skip=0
Status code: 200
Content type: application/json; charset=utf-8
{"metadata": {"skip":0,"select":null,"rundate":"2026-07-07T22:05:10.509Z","top":100,"filter":"","format":"json","metadata":true,"orderby":"","entityname":"FimaNfipClaims","version":"v2","url":"/api/open/v2/FimaNfipClaims?%24top=100&%24skip=0","count":0}, "FimaNfipClaims": [{"agricultureStructureIndi


In [3]:
def find_bucket_key(s3_path):
    """
    This is a helper function that given an s3 path such that the path is of
    the form: bucket/key
    It will return the bucket and the key represented by the s3 path
    """
    s3_components = s3_path.split('/')
    bucket = s3_components[0]
    s3_key = ""
    if len(s3_components) > 1:
        s3_key = '/'.join(s3_components[1:])
    return bucket, s3_key

In [4]:
import json
import requests
import urllib

# Convert API response into Python dict
data = response.json()

# Save JSON locally
local_file = "raw_fema_claims.json"

#opener = urllib.URLopener()

#myfile = opener.open(myurl)
#print(myfile)

with open(local_file, "w") as f:
    json.dump(data, f, indent=2)

# Upload to S3
s3.upload_file(
    Filename=local_file,
    Bucket=BUCKET_NAME,
    Key=s3_key
)

print("Uploaded file to bucket name/", {s3_key})

Uploaded file to bucket name/ {'raw/fema/unprocessed/fema_claims.json'}


In [5]:
s3.get_object(Bucket=BUCKET_NAME, Key=s3_key)

{'ResponseMetadata': {'RequestId': 'NZT48B599B3YCCE3',
  'HostId': '1KD5QUmLVVaZzHV6BT2dpmfoDdw0uE3F/u3PRFAr0s8kMSR/7xVT5fGU20E9vHg3dDI/oKbuwBQ=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': '1KD5QUmLVVaZzHV6BT2dpmfoDdw0uE3F/u3PRFAr0s8kMSR/7xVT5fGU20E9vHg3dDI/oKbuwBQ=',
   'x-amz-request-id': 'NZT48B599B3YCCE3',
   'date': 'Tue, 07 Jul 2026 22:05:12 GMT',
   'last-modified': 'Tue, 07 Jul 2026 22:05:11 GMT',
   'etag': '"d90ddcaac9642eb1e8439d6552113a63"',
   'x-amz-checksum-crc32': 'uW2QwA==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'x-amz-server-side-encryption': 'AES256',
   'x-amz-version-id': 'ExK.zqUmSecjypyokpV1zKnh4dh_TOus',
   'accept-ranges': 'bytes',
   'content-type': 'binary/octet-stream',
   'content-length': '276879',
   'server': 'AmazonS3'},
  'ChecksumAlgorithm': 'crc32',
  'RetryAttempts': 0},
 'AcceptRanges': 'bytes',
 'LastModified': datetime.datetime(2026, 7, 7, 22, 5, 11, tzinfo=tzutc()),
 'ContentLength': 276879,
 'ETag': '"d90ddcaac9642eb1e8439d

In [6]:
#print(data)
claims = data["FimaNfipClaims"]
df = pd.DataFrame(claims)
df.head()

,agricultureStructureIndicator,asOfDate,basementEnclosureCrawlspaceType,policyCount,crsClassificationCode,dateOfLoss,elevatedBuildingIndicator,elevationCertificateIndicator,elevationDifference,baseFloodElevation,...,rentalPropertyIndicator,state,reportedCity,reportedZipCode,countyCode,censusTract,censusBlockGroupFips,latitude,longitude,id
0,False,2026-06-01T00:00:00.000Z,NaN,1,NaN,1992-12-11T00:00:00.000Z,False,None,NaN,NaN,...,False,NJ,Currently Unavailable,07732,34025,34025800100,340258001002,40.4,-74.0,3f994197-be45-40fd-8abf-9afc13b2e1ea
1,False,2026-06-01T00:00:00.000Z,NaN,1,NaN,2018-10-10T00:00:00.000Z,True,None,6.0,7.4,...,False,FL,Currently Unavailable,32328,12037,12037970304,120379703041,29.7,-84.9,856f876b-ef99-4bdd-92d3-ab2d95cde7f3
2,False,2026-06-01T00:00:00.000Z,2.0,1,NaN,1996-12-16T00:00:00.000Z,False,None,NaN,NaN,...,False,PA,Currently Unavailable,19403,42091,42091203402,420912034022,40.1,-75.4,157d2c40-4505-4792-a0f9-773c8f0bdc59
3,False,2026-06-01T00:00:00.000Z,NaN,1,NaN,2001-06-14T00:00:00.000Z,False,2,NaN,NaN,...,False,MS,Currently Unavailable,39466,28109,28109950700,281099507001,30.5,-89.7,1e47522a-f4c1-4f1c-9505-124a74c472e8
4,False,2026-06-01T00:00:00.000Z,NaN,1,NaN,1979-07-26T00:00:00.000Z,False,None,NaN,NaN,...,False,TX,Currently Unavailable,77511,48039,None,None,29.4,-95.2,e6a616fb-8cbd-4d19-b13a-f67b64dd60e4


In [7]:
core_cols = [
    "id",
    "state",
    "yearOfLoss",
    "ratedFloodZone",
    "netBuildingPaymentAmount",
    "netContentsPaymentAmount",
    "totalBuildingInsuranceCoverage",
    "totalContentsInsuranceCoverage",
    "netIccPaymentAmount"
]

df_core = df[core_cols].copy()

df_core["totalPaidAmount"] = (
    df_core["netBuildingPaymentAmount"].fillna(0)
    + df_core["netContentsPaymentAmount"].fillna(0)
    + df_core["netIccPaymentAmount"].fillna(0)
)
df_core["totalPaidAmount"] = (
    df["netBuildingPaymentAmount"].fillna(0)
    + df["netContentsPaymentAmount"].fillna(0)
    + df["netIccPaymentAmount"].fillna(0)
)

In [8]:
top_10_states = (
    df_core.groupby("state", as_index=False)["totalPaidAmount"]
      .sum()
      .sort_values("totalPaidAmount", ascending=False)
)

top_10_states

,state,totalPaidAmount
4,FL,1729781.82
9,LA,831695.84
22,TX,388871.54
12,MS,289291.91
15,NJ,221127.01
20,SC,153143.63
11,MO,98790.22
7,IL,72103.45
16,NY,56767.61
14,ND,31318.35


In [9]:
state_analyst = (
    df.groupby("state", as_index=False)
      .agg(totalPolicyCount=("policyCount", "sum"))
      .sort_values("totalPolicyCount", ascending=False)
)

state_analyst

,state,totalPolicyCount
4,FL,27
9,LA,15
22,TX,10
11,MO,6
15,NJ,6
12,MS,4
16,NY,4
7,IL,3
20,SC,3
19,PA,3


In [10]:
import os

# Create the 'processed' directory if it doesn't already exist
os.makedirs('processed', exist_ok=True)

process_file = 'processed/state_analyst.csv'
state_analyst.to_csv(process_file, index=False)

try:
    s3.upload_file(Filename=process_file,Bucket=BUCKET_NAME, Key=s3_key)
    print(f"File {process_file} successfully uploaded to S3 bucket as {s3_key}")
except Exception as e:
    print(f"Error uploading file to S3: {e}")

File processed/state_analyst.csv successfully uploaded to S3 bucket as raw/fema/unprocessed/fema_claims.json
